## 1. Import dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
nltk.download('wordnet')


## 1. Importing the dataset
### 1.1 Load dataframe

In [ ]:
# Load the SMS Spam Collection dataset
filepath = "SMSSpamCollection"
df = pd.read_csv(filepath, sep="\t", header=None, names=["label", "message"])
df.head()


### 1.2 Drop uneccessary columns

In [ ]:
# There are only 2 columns, so nothing to drop.
df.info()


### 1.3 Check balance

In [ ]:
# Plot the distribution of 'ham' vs 'spam'
sns.countplot(x='label', data=df)
plt.title("Distribution of SMS labels (ham vs spam)")
plt.show()

print(df['label'].value_counts())


### 1.4 Trim size of dataframe (optional)

In [ ]:
print("Dataset shape:", df.shape)
# The dataset has about 5500 rows, which is perfectly fine to process fully without trimming.


## 2. Cleaning and pre-processing the data
### 2.1 Inspect data

In [ ]:
# Checking for missing values
print(df.isnull().sum())

# Convert labels to numerical scores: spam=1, ham=0
df['score'] = df['label'].map({'spam': 1, 'ham': 0})


### 2.3 Preprocessing all messages

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower() # Convert to lowercase
    text = re.sub('<.*?>', '', text) # Remove HTML tags 
    text = re.sub('[^a-zA-Z]', ' ', text) # Remove non-alphabetic
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

df['clean_message'] = df['message'].apply(preprocess_text)
df[['message', 'clean_message']].head()


## 3. Analyze the data
### 3.1 Inspect some data

In [ ]:
df.sample(5)[['message', 'clean_message', 'label']]


### 3.2 Analyze most common words

In [ ]:
corpus = ' '.join(df['clean_message'].tolist())
word_freq = Counter(corpus.split())
common_words = word_freq.most_common(20)

words, counts = zip(*common_words)
plt.figure(figsize=(10, 6))
sns.barplot(x=list(counts), y=list(words))
plt.title("Top 20 Most Common Words in SMSSpamCollection")
plt.show()


### 3.3 Word cloud of most common pos (Spam) words

In [ ]:
spam_corpus = ' '.join(df[df['label'] == 'spam']['clean_message'].tolist())
wordcloud_spam = WordCloud(width=800, height=400, background_color='white').generate(spam_corpus)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_spam, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud - Spam Messages")
plt.show()


### 3.4 Word cloud of most common neg (Ham) words

In [ ]:
ham_corpus = ' '.join(df[df['label'] == 'ham']['clean_message'].tolist())
wordcloud_ham = WordCloud(width=800, height=400, background_color='black').generate(ham_corpus)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_ham, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud - Ham Messages")
plt.show()


### 3.5 Comments

In [ ]:
print("Spam messages heavily feature words representing urgency and offers like 'call', 'free', 'txt', 'mobile', and 'claim'.")
print("Ham (regular) messages feature conversational words like 'u', 'go', 'get', 'come', 'ok'.")


## 4. Split the data into train & test

In [ ]:
X = df['clean_message']
y = df['score']

vectorizer = TfidfVectorizer(max_features=5000)
X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


### 4.2 Comments

In [ ]:
print("Data maps text features -> TF-IDF vectors representing 5000 top features.")
print("The data is split 80% (training) / 20% (testing).")


## 7. Naive Bayes

In [ ]:
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train, y_train)

y_pred = nb_classifier.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted (0:Ham, 1:Spam)")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Naive Bayes (Spam Classification)")
plt.show()


## 8. Test the Model

In [ ]:
# Test on some sample custom SMS messages
custom_sms = [
    "WINNER! claim your free iPhone now by calling exactly this number: 555-1234",
    "Hey hun, want to grab lunch around 1pm today? Let me know",
    "URGENT: Your bank account has been locked. Click here to verify your identity.",
    "Can you pick up milk on your way back home? Thanks"
]

custom_df = pd.DataFrame({'message': custom_sms})
custom_df['clean_message'] = custom_df['message'].apply(preprocess_text)
custom_vec = vectorizer.transform(custom_df['clean_message'])

custom_df['predicted_score'] = nb_classifier.predict(custom_vec)
custom_df['predicted_class'] = custom_df['predicted_score'].map({1: 'spam', 0: 'ham'})

display(custom_df[['message', 'predicted_class']])
